In [1]:
# === SESSION START: run this first, every time ===
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/projects/namibia-baseline-pack-demo"
%cd "{REPO_DIR}"

import sys, importlib
sys.path.insert(0, f"{REPO_DIR}/src")

import ee, geemap
try:
    ee.Initialize(project=userdata.get("EE_PROJECT"))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=userdata.get("EE_PROJECT"))
print("EE ready:", ee.String("ok").getInfo())

# --- state check: what is actually in the repo right now ---
!find . -path ./.git -prune -o -path ./outputs -prune -o -print | sort
!ls -lh outputs/rasters outputs/figures outputs/tables 2>/dev/null
!git status --short && git log --oneline -3

Mounted at /content/drive
/content/drive/MyDrive/projects/namibia-baseline-pack-demo
EE ready: ok
.
./config.yaml
./CREDITS.md
./data
./data/vector
./data/vector/aoi.gpkg
./data/vector/training.gpkg
./docs
./docs/class_key.md
./docs/time_log.md
./.gitignore
./LICENSE
./notebooks
./notebooks/05_rf_spectral.ipynb
./README.md
./requirements.txt
./src
./src/baseline_pack
./src/baseline_pack/aoi.py
./src/baseline_pack/composites.py
./src/baseline_pack/gitutils.py
./src/baseline_pack/__init__.py
./src/baseline_pack/__pycache__
./src/baseline_pack/__pycache__/aoi.cpython-313.pyc
./src/baseline_pack/__pycache__/composites.cpython-313.pyc
./src/baseline_pack/__pycache__/gitutils.cpython-313.pyc
./src/baseline_pack/__pycache__/__init__.cpython-313.pyc
./src/baseline_pack/__pycache__/trend.cpython-313.pyc
./src/baseline_pack/trend.py
outputs/figures:
total 214K
-rw------- 1 root root 214K Oct 21 09:58 ndvi_rain_2016_2025.png

outputs/rasters:
total 33M
-rw------- 1 root root 2.2M Oct 21 10:01 dry

In [2]:
from pathlib import Path
Path("notebooks").mkdir(exist_ok=True)
Path("notebooks/.gitkeep").touch()
!ls -la notebooks

total 10
drwx------ 2 root root 4096 Oct 22 07:29 .
drwx------ 8 root root 4096 Oct 21 07:10 ..
-rw------- 1 root root 1647 Oct 22 07:29 05_rf_spectral.ipynb
-rw------- 1 root root    0 Oct 22 07:29 .gitkeep


In [3]:
!git show --stat --oneline c93b76a

import subprocess, geopandas as gpd
def count_polys(path):
    try:
        return len(gpd.read_file(path, layer="training"))
    except Exception as e:
        return f"not readable ({type(e).__name__})"

for ref in ("origin/main", "HEAD"):
    blob = subprocess.run(["git", "show", f"{ref}:data/vector/training.gpkg"], capture_output=True).stdout
    tmp = f"/tmp/{ref.replace('/', '_')}.gpkg"
    open(tmp, "wb").write(blob)
    print(f"{ref:12s} →", count_polys(tmp) if blob else "file not in this version")
print(f"{'Drive copy':12s} →", count_polys("data/vector/training.gpkg"))

c93b76a (origin/main, origin/HEAD) Add files via upload
 data/vector/training.gpkg | Bin 0 -> 135168 bytes
 1 file changed, 0 insertions(+), 0 deletions(-)
origin/main  → 98
HEAD         → 100
Drive copy   → 100


In [4]:
from baseline_pack.gitutils import git_push
git_push("Training polygons (100) and Session 5 notebook")
!git log --oneline -3

git add → 
git commit → [main 4f2cc81] Training polygons (100) and Session 5 notebook
 2 files changed, 1 insertion(+)
 create mode 100644 notebooks/.gitkeep
 create mode 100644 notebooks/05_rf_spectral.ipynb
git push → warning: redirecting to https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git/
To https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git
   2c49d21..4f2cc81  HEAD -> main
4f2cc81 (HEAD -> main) Training polygons (100) and Session 5 notebook
2c49d21 Add training polygons (5-class key)
c93b76a (origin/main, origin/HEAD) Add files via upload


In [7]:
import yaml, geopandas as gpd, pandas as pd
import baseline_pack.classify as cl; importlib.reload(cl)
from baseline_pack.aoi import gdf_to_ee
from baseline_pack.composites import seasonal_composite, terrain

cfg = yaml.safe_load(open("config.yaml"))
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)

aoi = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi")).geometry()
region = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi_buffer")).geometry()
dry, _, _ = seasonal_composite(region, cfg["current_year"], cfg["dry_season_months"], "dry")
wet, _, _ = seasonal_composite(region, cfg["current_year"], cfg["wet_season_months"], "wet")
stack = cl.feature_stack(dry, wet, terrain(region))
FEATURES = stack.bandNames().getInfo()

train_gdf = cl.prepare_training(gpd.read_file("data/vector/training.gpkg", layer="training"), inner_buffer_m=5)
pts, balanced = cl.sample_pixels(stack, train_gdf)
print(len(FEATURES), "features;", len(train_gdf), "polygons left after 10 m inward buffer")
display(cl.pixel_counts(pts, CLASSES))

/content/drive/MyDrive/projects/namibia-baseline-pack-demo/src/baseline_pack/classify.py:21: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()


31 features; 98 polygons left after 10 m inward buffer


,train_px,test_px
class_id,,
1,108,25
2,247,73
3,254,72
4,488,222
5,412,76


In [8]:
import yaml
from pathlib import Path
cfg = yaml.safe_load(open("config.yaml"))
cfg["classes"][1]["name"] = "Closed woody vegetation"
cfg["classes"][2]["name"] = "Bush savanna"
yaml.safe_dump(cfg, open("config.yaml", "w"), sort_keys=False)

p = Path("docs/class_key.md")
t = p.read_text()
t = t.replace("| 1 | Dense bush | Woody canopy >= 40%; crowns mostly touching. Includes scattered taller trees. |",
              "| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |")
t = t.replace("| 2 | Open bush / shrub savanna | Woody canopy 10-40%; separate shrubs/trees with grass between. |",
              "| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |")
t += "\n## Change log\n- 2026-10-22: Class 1 threshold raised from 40% to 70% (closed woody vegetation) and class 2 widened to 10-70%, to match how training areas were actually interpreted.\n"
p.write_text(t)
print(p.read_text())

# Land-cover class key (reference year 2025)

Woody canopy cover = share of ground covered by tree and shrub crowns seen from above,
judged visually on high-resolution imagery. Height is not distinguished.

| ID | Class | Definition |
|---|---|---|
| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |
| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |
| 3 | Grassland / open | Woody canopy < 10%; includes cleared blocks now dominated by grass. |
| 4 | Bare / disturbed ground | Mostly exposed soil or rock: wide gravel tracks, quarries, fresh clearing, construction, dry pan floors. |
| 5 | Built-up | Buildings, paved surfaces and yards. |

## Decision rules for edge cases
- Scattered tall trees in grass: count their crowns; if total canopy < 10% -> Grassland / open.
- Cleared block w

In [9]:
!ls -la data/vector/
t = gpd.read_file("data/vector/training.gpkg", layer="training")
notes = t["notes"].fillna("")
print("Polygons:", len(t), "| new:", notes.str.contains("added 2026-10-22").sum(),
      "| reassigned:", notes.str.contains("reassigned").sum())
print(t.groupby("class_id").size().rename("polygons"))

total 260
drwx------ 2 root root   4096 Oct 21 07:11 .
drwx------ 3 root root   4096 Oct 21 07:11 ..
-rw------- 1 root root 118784 Oct 21 07:42 aoi.gpkg
-rw------- 1 root root 139264 Oct 22 08:06 training.gpkg
Polygons: 107 | new: 0 | reassigned: 0
class_id
1    26
2    28
3    19
4    17
5    17
Name: polygons, dtype: int64


In [10]:
import yaml
from pathlib import Path
cfg = yaml.safe_load(open("config.yaml"))
cfg["classes"][1]["name"] = "Closed woody vegetation"
cfg["classes"][2]["name"] = "Bush savanna"
yaml.safe_dump(cfg, open("config.yaml", "w"), sort_keys=False)

p = Path("docs/class_key.md")
t = p.read_text()
t = t.replace("| 1 | Dense bush | Woody canopy >= 40%; crowns mostly touching. Includes scattered taller trees. |",
              "| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |")
t = t.replace("| 2 | Open bush / shrub savanna | Woody canopy 10-40%; separate shrubs/trees with grass between. |",
              "| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |")
t += "\n## Change log\n- 2026-10-22: Class 1 threshold raised from 40% to 70% (closed woody vegetation) and class 2 widened to 10-70%, to match how training areas were actually interpreted.\n"
p.write_text(t)
print(p.read_text())

# Land-cover class key (reference year 2025)

Woody canopy cover = share of ground covered by tree and shrub crowns seen from above,
judged visually on high-resolution imagery. Height is not distinguished.

| ID | Class | Definition |
|---|---|---|
| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |
| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |
| 3 | Grassland / open | Woody canopy < 10%; includes cleared blocks now dominated by grass. |
| 4 | Bare / disturbed ground | Mostly exposed soil or rock: wide gravel tracks, quarries, fresh clearing, construction, dry pan floors. |
| 5 | Built-up | Buildings, paved surfaces and yards. |

## Decision rules for edge cases
- Scattered tall trees in grass: count their crowns; if total canopy < 10% -> Grassland / open.
- Cleared block w

In [14]:
import yaml, geopandas as gpd, pandas as pd
import baseline_pack.classify as cl; importlib.reload(cl)
from baseline_pack.aoi import gdf_to_ee
from baseline_pack.composites import seasonal_composite, terrain

cfg = yaml.safe_load(open("config.yaml"))
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)

aoi = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi")).geometry()
region = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi_buffer")).geometry()
dry, _, _ = seasonal_composite(region, cfg["current_year"], cfg["dry_season_months"], "dry")
wet, _, _ = seasonal_composite(region, cfg["current_year"], cfg["wet_season_months"], "wet")
stack = cl.feature_stack(dry, wet, terrain(region))
FEATURES = stack.bandNames().getInfo()

train_gdf = cl.prepare_training(gpd.read_file("data/vector/training.gpkg", layer="training"), inner_buffer_m=5)
pts, balanced = cl.sample_pixels(stack, train_gdf)
print(len(FEATURES), "features;", len(train_gdf), "polygons left after 5 m inward buffer")
display(cl.pixel_counts(pts, CLASSES))

/content/drive/MyDrive/projects/namibia-baseline-pack-demo/src/baseline_pack/classify.py:22: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  g["class_id"] = g["class_id"].astype(int)


31 features; 105 polygons left after 5 m inward buffer


,train_px,test_px
class_id,,
1,222,95
2,242,126
3,157,169
4,484,226
5,319,169


In [15]:
import inspect
print("Area-based split active:", "held" in inspect.getsource(cl.prepare_training))
share = train_gdf.assign(a=train_gdf.area).groupby(["class_id", "split"])["a"].sum().unstack()
print((100 * share["test"] / share.sum(axis=1)).round(0).rename("test_area_%"))

Area-based split active: True
class_id
1    30.0
2    34.0
3    52.0
4    31.0
5    35.0
Name: test_area_%, dtype: float64


In [16]:
git_push("Area-based holdout split, 5 m inner buffer")

git add → 
git commit → [main 16550e3] Area-based holdout split, 5 m inner buffer
 5 files changed, 94 insertions(+), 5 deletions(-)
 create mode 100644 src/baseline_pack/classify.py
git push → warning: redirecting to https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git/
To https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git
   4f2cc81..16550e3  HEAD -> main


In [17]:
train_s = balanced.filter(ee.Filter.eq("split", "train"))
test_s = balanced.filter(ee.Filter.eq("split", "test"))
clf_diag = cl.train_rf(train_s, FEATURES)

cm, per_class, oa = cl.holdout_check(clf_diag, test_s, CLASSES, NAMES)
print(f"Holdout overall accuracy (diagnostic only): {oa}%")
display(cm, per_class)

imp = pd.Series(clf_diag.explain().get("importance").getInfo()).sort_values(ascending=False)
display((100 * imp / imp.sum()).round(1).head(10).to_frame("importance_%"))

Holdout overall accuracy (diagnostic only): 59.8%


,Closed woody vegetation,Bush savanna,Grassland / open,Bare / disturbed ground,Built-up
Closed woody vegetation,85,10,0,0,0
Bush savanna,62,55,1,8,0
Grassland / open,0,0,7,130,13
Bare / disturbed ground,0,12,17,111,10
Built-up,0,0,0,7,143


,producer_acc_%,user_acc_%
Closed woody vegetation,89.5,57.8
Bush savanna,43.7,71.4
Grassland / open,4.7,28.0
Bare / disturbed ground,74.0,43.4
Built-up,95.3,86.1


,importance_%
wet_MNDWI,4.6
dry_B11,4.6
wet_B5,4.2
wet_B2,4.1
wet_B3,4.1
wet_B11,4.0
wet_B6,4.0
wet_B7,3.8
wet_B12,3.8
wet_B8,3.6


In [18]:
raw = gpd.read_file("data/vector/training.gpkg", layer="training")
test_pred = test_s.classify(clf_diag)
df_t = pd.DataFrame([f["properties"] for f in
                     test_pred.select(["poly_id", "class_id", "classification"]).getInfo()["features"]])

for cid in (3, 4):   # grassland and bare: per held-out polygon, what did the model predict?
    sub = df_t[df_t.class_id == cid]
    tab = pd.crosstab(sub.poly_id, sub.classification.map(NAMES))
    tab["notes"] = [raw.loc[train_gdf.index[train_gdf.poly_id == p][0], "notes"] for p in tab.index]
    print(f"\nHeld-out {NAMES[cid]} polygons:"); display(tab)

suspects = train_gdf[train_gdf.class_id.isin([3, 4]) & (train_gdf.split == "test")]
Map = geemap.Map(basemap="HYBRID")
Map.addLayer(wet, {"bands": ["wet_B8", "wet_B4", "wet_B3"], "min": 0, "max": 0.35}, "Wet 2025 false colour")
Map.addLayer(dry, {"bands": ["dry_B8", "dry_B4", "dry_B3"], "min": 0, "max": 0.35}, "Dry 2025 false colour", False)
Map.addLayer(stack.classify(clf_diag), {"min": 1, "max": 5, "palette": PALETTE}, "Diagnostic map", False)
Map.addLayer(gdf_to_ee(suspects[suspects.class_id == 3]).style(color="yellow", fillColor="00000000", width=2), {}, "Held-out grassland")
Map.addLayer(gdf_to_ee(suspects[suspects.class_id == 4]).style(color="orange", fillColor="00000000", width=2), {}, "Held-out bare")
Map.addLayer(gdf_to_ee(train_gdf[(train_gdf.class_id == 3) & (train_gdf.split == "train")])
             .style(color="lime", fillColor="00000000", width=2), {}, "Training grassland")
Map.centerObject(aoi, 12); Map


Held-out Grassland / open polygons:


classification,Bare / disturbed ground,Built-up,Grassland / open,notes
poly_id,,,,
27,0,0,4,None
28,0,0,3,None
58,0,13,0,None
61,6,0,0,None
62,5,0,0,None
76,119,0,0,None



Held-out Bare / disturbed ground polygons:


classification,Bare / disturbed ground,Built-up,Bush savanna,Grassland / open,notes
poly_id,,,,,
29,4,0,0,0,None
30,4,0,0,3,None
63,19,0,0,0,None
68,2,0,0,0,None
70,0,10,0,0,None
72,82,0,0,0,None
75,0,0,12,14,None


Map(center=[-20.549999280997785, 16.56999870663687], controls=(WidgetControl(options=['position', 'transparent…

In [19]:
from pathlib import Path
p = Path("docs/class_key.md"); t = p.read_text()
swaps = {
 "| 3 | Grassland / open | Woody canopy < 10%; includes cleared blocks now dominated by grass. |":
 "| 3 | Grassland / open | Woody canopy < 10% and ground mostly covered by herbaceous vegetation in the wet season (red in the wet 2025 false colour). Includes cleared blocks where grass has re-established. |",
 "| 4 | Bare / disturbed ground | Mostly exposed soil or rock: wide gravel tracks, quarries, fresh clearing, construction, dry pan floors. |":
 "| 4 | Bare / disturbed ground | Woody canopy < 10% and mostly exposed soil or rock even in the wet season (grey/beige/white in the wet 2025 false colour): quarries, construction, fresh clearing, wide gravel areas, pan floors. |",
 "- Pans: bare floor in the 2025 dry season -> Bare; grass-covered -> Grassland / open.":
 "- Grass vs bare is decided on the wet-season (Dec 2024-Mar 2025) image, never on the dry season alone.\n- Pans: grass-covered in the wet season -> Grassland / open; otherwise -> Bare.",
}
for old, new in swaps.items():
    print("found" if old in t else "NOT FOUND", "→", old[:45])
    t = t.replace(old, new)
t += "- 2026-10-22: Grassland vs bare redefined by wet-season vegetation cover; classes 3 and 4 re-digitised.\n"
p.write_text(t)

found → | 3 | Grassland / open | Woody canopy < 10%; 
found → | 4 | Bare / disturbed ground | Mostly expose
found → - Pans: bare floor in the 2025 dry season -> 


2404

In [20]:
t = gpd.read_file("data/vector/training.gpkg", layer="training")
print("Polygons:", len(t), "| redrawn:", t["notes"].fillna("").str.contains("redrawn").sum())
print(t.groupby("class_id").agg(n=("class_id", "size"), ha=("geometry", lambda g: round(g.area.sum() / 1e4, 1))))
git_push("Re-digitise grassland and bare with wet-season rule; update class key")

Polygons: 104 | redrawn: 0
           n    ha
class_id          
1.0       26   5.3
2.0       28   5.8
3.0       17  43.3
4.0       15  31.1
5.0       17   7.0
git add → 
git commit → [main 1b8d3db] Re-digitise grassland and bare with wet-season rule; update class key
 3 files changed, 6 insertions(+), 4 deletions(-)
git push → warning: redirecting to https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git/
To https://github.com/ybrosig-geo/namibia-baseline-pack-demo.git
   16550e3..1b8d3db  HEAD -> main


In [21]:
t = gpd.read_file("data/vector/training.gpkg", layer="training")
t["ha"] = t.area / 1e4
print("Missing class_id:", t["class_id"].isna().sum(),
      "| redrawn:", t["notes"].fillna("").str.contains("redrawn").sum())
display(t.groupby("class_id")["ha"].describe()[["count", "min", "50%", "max"]].round(2))


Missing class_id: 1 | redrawn: 0


,count,min,50%,max
class_id,,,,
1.0,26.0,0.02,0.11,0.99
2.0,28.0,0.02,0.14,0.65
3.0,17.0,0.29,0.83,16.02
4.0,15.0,0.32,1.75,5.72
5.0,17.0,0.03,0.18,1.44


In [22]:
t = gpd.read_file("data/vector/training.gpkg", layer="training")
bad = t[t["class_id"].isna()]
c = bad.to_crs(4326).geometry.centroid.iloc[0]
print(f"Index {bad.index[0]} | area {bad.area.iloc[0]/1e4:.2f} ha | lon {c.x:.5f}, lat {c.y:.5f}")

Map = geemap.Map(basemap="HYBRID")
Map.addLayer(wet, {"bands": ["wet_B8", "wet_B4", "wet_B3"], "min": 0, "max": 0.35}, "Wet 2025 false colour")
Map.addLayer(dry, {"bands": ["dry_B8", "dry_B4", "dry_B3"], "min": 0, "max": 0.35}, "Dry 2025 false colour", False)
Map.addLayer(gdf_to_ee(bad).style(color="magenta", fillColor="00000000", width=3), {}, "Unlabelled polygon")
Map.setCenter(c.x, c.y, 16); Map

Index 94 | area 0.54 ha | lon 16.61984, lat -20.55124


Map(center=[-20.55123976544116, 16.61984228435765], controls=(WidgetControl(options=['position', 'transparent_…

In [23]:
import os
CLASS_FOR_IT = 3            # <-- put the class you decided on
t.loc[bad.index, "class_id"] = CLASS_FOR_IT
t["class_id"] = t["class_id"].astype(int)
t.to_file("data/vector/training_tmp.gpkg", layer="training", driver="GPKG")
os.replace("data/vector/training_tmp.gpkg", "data/vector/training.gpkg")
print("Missing now:", gpd.read_file("data/vector/training.gpkg", layer="training")["class_id"].isna().sum())

Missing now: 0


In [24]:
import yaml, geopandas as gpd, pandas as pd
import baseline_pack.classify as cl; importlib.reload(cl)
from baseline_pack.aoi import gdf_to_ee
from baseline_pack.composites import seasonal_composite, terrain

cfg = yaml.safe_load(open("config.yaml"))
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)

aoi = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi")).geometry()
region = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi_buffer")).geometry()
dry, _, _ = seasonal_composite(region, cfg["current_year"], cfg["dry_season_months"], "dry")
wet, _, _ = seasonal_composite(region, cfg["current_year"], cfg["wet_season_months"], "wet")
stack = cl.feature_stack(dry, wet, terrain(region))
FEATURES = stack.bandNames().getInfo()

train_gdf = cl.prepare_training(gpd.read_file("data/vector/training.gpkg", layer="training"))
pts, balanced = cl.sample_pixels(stack, train_gdf)
print(len(FEATURES), "features;", len(train_gdf), "polygons left after 10 m inward buffer")
display(cl.pixel_counts(pts, CLASSES))

31 features; 104 polygons left after 10 m inward buffer


,train_px,test_px
class_id,,
1,222,95
2,242,126
3,1860,2092
4,1814,798
5,333,155


In [25]:
print("Pixels used per class:", balanced.aggregate_histogram("class_id").getInfo())

KeyboardInterrupt: 

In [26]:
train_s = balanced.filter(ee.Filter.eq("split", "train"))
test_s = balanced.filter(ee.Filter.eq("split", "test"))
clf_diag = cl.train_rf(train_s, FEATURES)

cm, per_class, oa = cl.holdout_check(clf_diag, test_s, CLASSES, NAMES)
print(f"Holdout overall accuracy (diagnostic only): {oa}%")
display(cm, per_class)

imp = pd.Series(clf_diag.explain().get("importance").getInfo()).sort_values(ascending=False)
display((100 * imp / imp.sum()).round(1).head(10).to_frame("importance_%"))

KeyboardInterrupt: 

In [35]:
import hashlib, time
import baseline_pack.samples as sp; importlib.reload(sp)

fp = hashlib.md5(open("data/vector/training.gpkg", "rb").read()).hexdigest()[:8]
CSV = f"outputs/tables/training_pixels_2025_{fp}.csv"
if Path(CSV).exists():
    px = pd.read_csv(CSV); print("Loaded saved pixels:", CSV)
else:
    print("Extracting pixels (one-time background export, a few minutes)…")
    px = sp.extract_pixels_to_csv(stack, train_gdf, CSV, name=f"training_pixels_{fp}")

px = px.dropna(subset=FEATURES)
cfg = yaml.safe_load(open("config.yaml"))
REMAP = {int(k): v for k, v in cfg["label_remap"].items()}
px["class_id"] = px["class_id"].map(REMAP)
train_gdf["class_id"] = train_gdf["class_id"].map(REMAP)
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)
bal = sp.balance(px)
print("All pixels in polygons:"); display(px.groupby(["class_id", "split"]).size().unstack())
print("Pixels used by the model:"); display(bal.groupby(["class_id", "split"]).size().unstack())

balanced = sp.to_fc(bal, FEATURES)     # replaces the old server-side 'balanced'

Loaded saved pixels: outputs/tables/training_pixels_2025_ca7fc150.csv
All pixels in polygons:


split,test,train
class_id,,
1,95,222
2,126,242
3,2890,3674
4,155,333


Pixels used by the model:


split,test,train
class_id,,
1,95,163
2,126,228
3,150,300
4,95,197


In [36]:
train_s = balanced.filter(ee.Filter.eq("split", "train"))
test_s = balanced.filter(ee.Filter.eq("split", "test"))
clf_diag = cl.train_rf(train_s, FEATURES)

cm, per_class, oa = cl.holdout_check(clf_diag, test_s, CLASSES, NAMES)
print(f"Holdout overall accuracy (diagnostic only): {oa}%")
display(cm, per_class)

imp = pd.Series(clf_diag.explain().get("importance").getInfo()).sort_values(ascending=False)
display((100 * imp / imp.sum()).round(1).head(10).to_frame("importance_%"))

Holdout overall accuracy (diagnostic only): 80.9%


,Closed woody vegetation,Bush savanna,Open land (grass or bare),Built-up
Closed woody vegetation,84,11,0,0
Bush savanna,56,61,9,0
Open land (grass or bare),0,8,141,1
Built-up,0,0,4,91


,producer_acc_%,user_acc_%
Closed woody vegetation,88.4,60.0
Bush savanna,48.4,76.2
Open land (grass or bare),94.0,91.6
Built-up,95.8,98.9


,importance_%
elevation,5.6
wet_B2,5.5
wet_B3,5.0
wet_B12,4.5
wet_B11,4.1
dry_B2,3.7
dry_B3,3.7
wet_B8A,3.7
dry_NDVI,3.4
wet_B5,3.4


In [32]:
cols = ["wet_NDVI", "dry_NDVI", "dNDVI", "wet_BSI", "dry_BSI"]
q = px.groupby("class_id")[cols].quantile([0.1, 0.5, 0.9]).unstack().round(2)
q.index = [NAMES[i] for i in q.index]
print("Feature values per class (10th / 50th / 90th percentile):")
display(q["wet_NDVI"], q["dNDVI"], q["dry_NDVI"])

# per polygon: median wet NDVI for grassland and bare
poly = (px[px.class_id.isin([3, 4])].groupby(["class_id", "poly_id"])["wet_NDVI"].median()
        .reset_index().sort_values(["class_id", "wet_NDVI"]))
g_med = poly[poly.class_id == 3].wet_NDVI.median(); b_med = poly[poly.class_id == 4].wet_NDVI.median()
poly["suspicious"] = ((poly.class_id == 4) & (poly.wet_NDVI > g_med)) | ((poly.class_id == 3) & (poly.wet_NDVI < b_med))
print(f"Median wet NDVI: grassland polygons {g_med:.2f}, bare polygons {b_med:.2f}")
display(poly.round(2))

sus = train_gdf[train_gdf.poly_id.isin(poly[poly.suspicious].poly_id)]
Map = geemap.Map(basemap="HYBRID")
Map.addLayer(wet, {"bands": ["wet_B8", "wet_B4", "wet_B3"], "min": 0, "max": 0.35}, "Wet 2025 false colour")
Map.addLayer(gdf_to_ee(train_gdf[train_gdf.class_id == 3]).style(color="yellow", fillColor="00000000", width=2), {}, "Grassland")
Map.addLayer(gdf_to_ee(train_gdf[train_gdf.class_id == 4]).style(color="orange", fillColor="00000000", width=2), {}, "Bare")
if len(sus):
    Map.addLayer(gdf_to_ee(sus).style(color="magenta", fillColor="00000000", width=3), {}, "Suspicious")
Map.centerObject(aoi, 12); Map

Feature values per class (10th / 50th / 90th percentile):


,0.1,0.5,0.9
Closed woody vegetation,0.35,0.55,0.71
Bush savanna,0.23,0.31,0.48
Grassland / open,0.17,0.22,0.31
Bare / disturbed ground,0.08,0.16,0.23
Built-up,-0.00,0.04,0.14


,0.1,0.5,0.9
Closed woody vegetation,0.02,0.14,0.30
Bush savanna,-0.05,0.06,0.25
Grassland / open,-0.01,0.06,0.14
Bare / disturbed ground,-0.06,0.00,0.06
Built-up,-0.03,-0.01,0.01


,0.1,0.5,0.9
Closed woody vegetation,0.22,0.39,0.54
Bush savanna,0.16,0.24,0.31
Grassland / open,0.15,0.16,0.19
Bare / disturbed ground,0.07,0.17,0.20
Built-up,0.00,0.05,0.15


Median wet NDVI: grassland polygons 0.25, bare polygons 0.16


,class_id,poly_id,wet_NDVI,suspicious
1,3,73,0.18,False
13,3,99,0.19,False
17,3,103,0.20,False
16,3,102,0.21,False
5,3,82,0.22,False
6,3,83,0.22,False
12,3,98,0.23,False
8,3,94,0.24,False
0,3,71,0.25,False
3,3,76,0.26,False


Map(center=[-20.549999280998136, 16.569998706636877], controls=(WidgetControl(options=['position', 'transparen…

In [33]:
p100 = train_gdf[train_gdf.poly_id == 100]
Map.addLayer(gdf_to_ee(p100).style(color="cyan", fillColor="00000000", width=3), {}, "Polygon 100")
Map.centerObject(gdf_to_ee(p100), 16); Map

Map(bottom=585770.0, center=[-20.463234957506096, 16.642287085669693], controls=(WidgetControl(options=['posit…

In [34]:
import yaml
from pathlib import Path
cfg = yaml.safe_load(open("config.yaml"))
cfg["label_remap"] = {1: 1, 2: 2, 3: 3, 4: 3, 5: 4}     # digitised label -> map class
cfg["classes"] = {
    1: {"name": "Closed woody vegetation",   "color": "#1b5e20"},
    2: {"name": "Bush savanna",              "color": "#8bc34a"},
    3: {"name": "Open land (grass or bare)", "color": "#e8d98a"},
    4: {"name": "Built-up",                  "color": "#d32f2f"},
}
yaml.safe_dump(cfg, open("config.yaml", "w"), sort_keys=False)

Path("docs/class_key.md").write_text("""# Land-cover class key (reference year 2025)

Woody canopy cover = share of ground covered by tree and shrub crowns seen from above,
judged visually on high-resolution imagery and 2025 Sentinel-2 false colour. Height is not distinguished.

| ID | Class | Definition |
|---|---|---|
| 1 | Closed woody vegetation | Woody canopy >= 70%; crowns nearly continuous, little or no ground visible. Thickets, hill slopes, drainage lines; often includes taller trees. |
| 2 | Bush savanna | Woody canopy 10-70%; distinct shrubs/trees with ground or grass visible between them. Most encroached rangeland. |
| 3 | Open land (grass or bare) | Woody canopy < 10%: grassland, sparse grass, exposed soil or rock, cleared ground, pans, wide gravel areas. |
| 4 | Built-up | Buildings, paved surfaces and yards. |

## Decision rules
- Scattered trees in open land: count their crowns; total canopy < 10% -> Open land.
- Cleared blocks: judged as they are in 2025, not before clearing.
- Paved roads in town -> Built-up; gravel tracks narrower than ~10 m are not mapped separately.
- Mixed areas: the class covering the majority of the area being judged.

## Excluded on purpose
- Water: pans are seasonal; surface water is mapped separately as occurrence (pack item 4).
- Cropland / irrigated land: none present in the AOI in 2025.
- Tall-tree woodland as its own class: not reliably separable from dense bush at 10 m.

## Change log
- 2026-10-22: Class 1 threshold raised from 40% to 70% and class 2 widened to 10-70%, to match how training areas were interpreted.
- 2026-10-22: Grassland and bare ground were digitised separately but merged into 'Open land'. Their wet-season NDVI overlapped strongly
  (grassland polygons 0.18-0.37, bare 0.08-0.24), so they were not reliably separable at 10 m. Original labels are kept in training.gpkg;
  clearing and bare-ground change is reported in the disturbance footprint instead.
""")
print(open("config.yaml").read())

epsg: 32733
aoi_file: data/vector/aoi.gpkg
buffer_m: 2000
current_year: 2025
trend_years:
- 2016
- 2025
disturbance_reference_year: 2016
dry_season_months:
- 6
- 9
wet_season_months:
- 12
- 3
rain_year_start_month: 6
classes:
  1:
    name: Closed woody vegetation
    color: '#1b5e20'
  2:
    name: Bush savanna
    color: '#8bc34a'
  3:
    name: Open land (grass or bare)
    color: '#e8d98a'
  4:
    name: Built-up
    color: '#d32f2f'
label_remap:
  1: 1
  2: 2
  3: 3
  4: 3
  5: 4



In [38]:
poly12 = (px[px.class_id.isin([1, 2])].groupby(["class_id", "poly_id"])[["dry_NDVI", "wet_NDVI"]]
          .median().round(2).reset_index().sort_values(["class_id", "dry_NDVI"]))
display(poly12)
print(px[px.class_id.isin([1, 2])].groupby("class_id")["dry_NDVI"].quantile([0.1, 0.5, 0.9]).unstack().round(2))

,class_id,poly_id,dry_NDVI,wet_NDVI
16,1,60,0.18,0.46
15,1,59,0.21,0.45
22,1,68,0.23,0.35
8,1,51,0.27,0.72
13,1,56,0.28,0.60
7,1,50,0.29,0.47
19,1,63,0.31,0.37
0,1,1,0.33,0.60
1,1,44,0.34,0.58
6,1,49,0.34,0.54


           0.1   0.5   0.9
class_id                  
1         0.22  0.39  0.54
2         0.16  0.24  0.31


In [39]:
fp = hashlib.md5(open("data/vector/training.gpkg", "rb").read() + ",".join(FEATURES).encode()).hexdigest()[:8]

In [43]:
import hashlib, yaml
from pathlib import Path
import baseline_pack.samples as sp; importlib.reload(sp)

cfg = yaml.safe_load(open("config.yaml"))
REMAP = {int(k): v for k, v in cfg["label_remap"].items()}
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)

# Always start from the original digitised labels -> safe to rerun this cell any time
train_gdf = cl.prepare_training(gpd.read_file("data/vector/training.gpkg", layer="training"))

# Fingerprint = training polygons + feature list. Change either -> new extraction runs automatically
fp = hashlib.md5(open("data/vector/training.gpkg", "rb").read() + ",".join(FEATURES).encode()).hexdigest()[:8]
CSV = f"outputs/tables/training_pixels_2025_{fp}.csv"
if Path(CSV).exists():
    px = pd.read_csv(CSV); print("Loaded saved pixels:", CSV)
else:
    print(f"Extracting pixels for {len(FEATURES)} features (background export, a few minutes)…")
    px = sp.extract_pixels_to_csv(stack, train_gdf, CSV, name=f"training_pixels_{fp}")

px = px.dropna(subset=FEATURES)
px["class_id"] = px["class_id"].map(REMAP)              # merge grassland + bare -> open land
train_gdf["class_id"] = train_gdf["class_id"].map(REMAP)

bal = sp.balance(px)
print("Pixels used by the model:"); display(bal.groupby(["class_id", "split"]).size().unstack())
balanced = sp.to_fc(bal, FEATURES)

Loaded saved pixels: outputs/tables/training_pixels_2025_8ebd02b2.csv
Pixels used by the model:


split,test,train
class_id,,
1,95,163
2,126,228
3,150,300
4,95,197


In [41]:
import inspect
importlib.reload(cl)
print("File used:", cl.__file__)
print("Texture version active:", "dry_NDVI_sd3" in inspect.getsource(cl.feature_stack))

File used: /content/drive/MyDrive/projects/namibia-baseline-pack-demo/src/baseline_pack/classify.py
Texture version active: True


In [44]:
# === SESSION 5: features -> training pixels -> diagnostic Random Forest (all in one) ===
import importlib, hashlib, yaml
import pandas as pd, geopandas as gpd
from pathlib import Path
import baseline_pack.classify as cl; importlib.reload(cl)
import baseline_pack.samples as sp; importlib.reload(sp)
from baseline_pack.aoi import gdf_to_ee
from baseline_pack.composites import seasonal_composite, terrain

# settings and class names (4-class scheme)
cfg = yaml.safe_load(open("config.yaml"))
REMAP = {int(k): v for k, v in cfg["label_remap"].items()}
NAMES = {int(k): v["name"] for k, v in cfg["classes"].items()}
PALETTE = [cfg["classes"][k]["color"].lstrip("#") for k in sorted(cfg["classes"])]
CLASSES = sorted(NAMES)

# composites + feature stack (with texture)
aoi = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi")).geometry()
region = gdf_to_ee(gpd.read_file("data/vector/aoi.gpkg", layer="aoi_buffer")).geometry()
dry, _, _ = seasonal_composite(region, cfg["current_year"], cfg["dry_season_months"], "dry")
wet, _, _ = seasonal_composite(region, cfg["current_year"], cfg["wet_season_months"], "wet")
stack = cl.feature_stack(dry, wet, terrain(region))
FEATURES = stack.bandNames().getInfo()
print(len(FEATURES), "features")
assert len(FEATURES) == 33, "Texture bands missing - stop here and tell me"

# training pixels: load saved CSV, or extract once (background export)
train_gdf = cl.prepare_training(gpd.read_file("data/vector/training.gpkg", layer="training"))
fp = hashlib.md5(open("data/vector/training.gpkg", "rb").read() + ",".join(FEATURES).encode()).hexdigest()[:8]
CSV = f"outputs/tables/training_pixels_2025_{fp}.csv"
if Path(CSV).exists():
    px = pd.read_csv(CSV); print("Loaded saved pixels:", CSV)
else:
    print("Extracting pixels (background export, a few minutes)…")
    px = sp.extract_pixels_to_csv(stack, train_gdf, CSV, name=f"training_pixels_{fp}")

px = px.dropna(subset=FEATURES)
px["class_id"] = px["class_id"].map(REMAP)
train_gdf["class_id"] = train_gdf["class_id"].map(REMAP)
bal = sp.balance(px)
balanced = sp.to_fc(bal, FEATURES)

# diagnostic Random Forest: train on 70% of polygons, test on the other 30%
train_s = balanced.filter(ee.Filter.eq("split", "train"))
test_s = balanced.filter(ee.Filter.eq("split", "test"))
clf_diag = cl.train_rf(train_s, FEATURES)
cm, per_class, oa = cl.holdout_check(clf_diag, test_s, CLASSES, NAMES)

print(f"\nHoldout overall accuracy (diagnostic only): {oa}%")
display(cm, per_class)
imp = pd.Series(clf_diag.explain().get("importance").getInfo()).sort_values(ascending=False)
display((100 * imp / imp.sum()).round(1).head(10).to_frame("importance_%"))

33 features
Extracting pixels (background export, a few minutes)…
Export running… READY

Holdout overall accuracy (diagnostic only): 83.0%


,Closed woody vegetation,Bush savanna,Open land (grass or bare),Built-up
Closed woody vegetation,85,10,0,0
Bush savanna,49,68,9,0
Open land (grass or bare),0,8,141,1
Built-up,0,0,2,93


,producer_acc_%,user_acc_%
Closed woody vegetation,89.5,63.4
Bush savanna,54.0,79.1
Open land (grass or bare),94.0,92.8
Built-up,97.9,98.9


,importance_%
elevation,5.3
wet_B2,5.0
wet_B12,4.0
dry_NDVI_sd5,4.0
wet_B3,3.8
dry_B5,3.4
dry_B11,3.4
wet_B5,3.4
wet_B4,3.4
wet_MNDWI,3.3


In [ ]:
# === SESSION 5: final model -> map v1 -> export -> area table -> comparison map ===
import time, glob, shutil, numpy as np, rasterio
from rasterio.mask import mask

# 1. final model on all sampled pixels, classify
clf = cl.train_rf(balanced, FEATURES)
lc_v1 = stack.classify(clf).rename("class").toByte()

# 2. background export to Drive, then copy into the repo outputs
name = "landcover_2025_rf_spectral_v1"
task = ee.batch.Export.image.toDrive(image=lc_v1, description=name, folder="baseline_pack_exports",
                                     fileNamePrefix=name, region=region, scale=10,
                                     crs="EPSG:32733", maxPixels=1e9)
task.start()
while task.active():
    print("Export running…", task.status()["state"]); time.sleep(30)
print("Export:", task.status()["state"])
hits = []
for _ in range(20):
    hits = glob.glob(f"/content/drive/MyDrive/baseline_pack_exports/{name}*.tif")
    if hits: break
    time.sleep(15)
out_tif = f"outputs/rasters/{name}.tif"
shutil.copy(hits[0], out_tif); print("Saved:", out_tif)

# 3. mapped area per class inside the AOI (pixel counting, NOT the final reported area)
aoi_gdf = gpd.read_file("data/vector/aoi.gpkg", layer="aoi")
with rasterio.open(out_tif) as src:
    arr, _ = mask(src, aoi_gdf.geometry, crop=True, nodata=0)
    px_ha = abs(src.res[0] * src.res[1]) / 1e4
vals, counts = np.unique(arr[arr > 0], return_counts=True)
areas_v1 = pd.DataFrame({"class_name": [NAMES[int(v)] for v in vals],
                         "area_ha": (counts * px_ha).round(1)}, index=vals.astype(int))
areas_v1["percent"] = (100 * areas_v1["area_ha"] / areas_v1["area_ha"].sum()).round(1)
areas_v1.to_excel("outputs/tables/landcover_2025_v1_mapped_area.xlsx")
display(areas_v1)

# 4. comparison map
wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map").clip(region)
dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(region)
      .filterDate("2025-01-01", "2026-01-01").select("label").mode().clip(region))
DW_PAL = ["419bdf", "397d49", "88b053", "7a87c6", "e49635", "dfc35a", "c4281b", "a59b8f", "b39fe1"]
Map = geemap.Map(basemap="HYBRID")
Map.addLayer(dry, {"bands": ["dry_B8", "dry_B4", "dry_B3"], "min": 0, "max": 0.35}, "Dry 2025 false colour", False)
Map.addLayer(wc, {}, "WorldCover 2021", False)
Map.addLayer(dw, {"min": 0, "max": 8, "palette": DW_PAL}, "Dynamic World 2025", False)
Map.addLayer(lc_v1, {"min": 1, "max": 4, "palette": PALETTE}, "Land cover v1")
Map.addLayer(gdf_to_ee(aoi_gdf).style(color="cyan", fillColor="00000000", width=2), {}, "AOI")
Map.add_legend(title="Land cover v1", labels=[NAMES[c] for c in CLASSES], colors=["#" + p for p in PALETTE])
Map.centerObject(aoi, 12)
display(Map)

# 5. save code + notebook to GitHub
from baseline_pack.gitutils import git_push
git_push("Session 5: RF land cover v1 (4 classes, texture), mapped areas")

Export running… READY
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING
Export running… RUNNING


Export running… RUNNING
Export: COMPLETED
Saved: outputs/rasters/landcover_2025_rf_spectral_v1.tif


,class_name,area_ha,percent
1,Closed woody vegetation,2730.0,6.8
2,Bush savanna,21158.8,52.9
3,Open land (grass or bare),15611.4,39.0
4,Built-up,499.8,1.2


TimeoutException: Requesting secret GOOGLE_MAPS_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.